# Module 10 - Session 3: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To gain practical experience using and interpreting the outputs of XAI libraries and to think critically about accountability.


## Setup

Install required libraries:

```bash
pip install scikit-learn pandas lime shap matplotlib seaborn
```

Dataset source (Adult Census Income):
- https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data


## Exercise 1: Conceptual Questions (30 minutes)

### 1) LIME vs. SHAP
LIME and SHAP answer different explanation questions. LIME asks: *"What simple local model best approximates this complex model around one specific instance?"* It explains behavior via a local surrogate fit in a neighborhood. SHAP asks: *"How much did each feature contribute to this prediction relative to a baseline expectation?"* It assigns additive contributions using Shapley-value logic, so feature attributions sum to the prediction shift.

### 2) The "Right to Explanation"
A bank can use LIME/SHAP to generate an instance-level explanation for a denied application, showing the top factors that pushed the decision toward denial. Example customer-facing explanation: "Your application was declined mainly because debt-to-income ratio was high, recent credit utilization was high, and credit history length was short; stable employment length partially improved your score, but not enough to pass the approval threshold." This does not reveal proprietary model internals but provides understandable reason codes.

### 3) Accountability in Open Source
Primary accountability lies with the data scientist and especially the deploying company, not the open-source library maintainers. The library provides general tools; deployment decisions (data selection, labeling, feature design, thresholds, governance, and monitoring) are made by the organization using the model. Open-source developers should follow good engineering practice, but legal and ethical responsibility for harmful outcomes in production belongs to the institution that built and deployed the system.


## Exercise 2: Explaining a Prediction with LIME (45 minutes)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from lime.lime_tabular import LimeTabularExplainer

sns.set_theme(style='whitegrid')


In [ ]:
# Load Adult dataset
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, header=None, names=columns, na_values=' ?', skipinitialspace=True)

# Binary target: 1 for >50K, 0 for <=50K
df['target'] = (df['income'] == '>50K').astype(int)
X = df.drop(columns=['income', 'target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample'
)

clf = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

clf.fit(X_train, y_train)
preds = clf.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, preds, target_names=['<=50K', '>50K']))


In [ ]:
# Build transformed matrices for LIME (LIME expects numeric matrix + feature names)
X_train_enc = clf.named_steps['prep'].transform(X_train)
X_test_enc = clf.named_steps['prep'].transform(X_test)

# Convert sparse matrix to dense for LIME compatibility
if hasattr(X_train_enc, 'toarray'):
    X_train_enc = X_train_enc.toarray()
if hasattr(X_test_enc, 'toarray'):
    X_test_enc = X_test_enc.toarray()

feature_names = clf.named_steps['prep'].get_feature_names_out()
class_names = ['<=50K', '>50K']

# Model-only probability function for already-transformed vectors
rf_model = clf.named_steps['model']
predict_proba_encoded = lambda data: rf_model.predict_proba(data)

# Find one misclassified test instance
wrong_idx = np.where(preds != y_test.to_numpy())[0]
print('Misclassified samples in test set:', len(wrong_idx))
idx = int(wrong_idx[0])
print('Chosen test index:', idx)
print('True label:', y_test.to_numpy()[idx], '| Pred label:', preds[idx])


In [ ]:
# Create LIME explainer and explain one wrong prediction
explainer = LimeTabularExplainer(
    training_data=X_train_enc,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

exp = explainer.explain_instance(
    data_row=X_test_enc[idx],
    predict_fn=predict_proba_encoded,
    num_features=10,
    top_labels=1
)

fig = exp.as_pyplot_figure(label=exp.top_labels[0])
plt.title('LIME Explanation for One Misclassified Individual')
plt.tight_layout()
plt.show()

print('Top LIME contributions:')
for feat, w in exp.as_list(label=exp.top_labels[0]):
    direction = 'pushes to >50K' if w > 0 else 'pushes to <=50K'
    print(f'{feat:60s} {w:+.4f} ({direction})')


### LIME Analysis
For the selected misclassified individual, LIME shows which encoded feature conditions most strongly pushed the model toward the predicted class. The mistake usually happens when several strong local signals (for example, high `capital-gain`, certain marital/occupation categories, or long work hours) outweigh counter-signals for that specific person. In other words, the model followed the dominant local pattern it learned from training data, but for this edge case that pattern did not match the true label.


## Exercise 3: Challenge Problem - Global Explanations with SHAP


In [ ]:
import shap

# SHAP can work with the tree model on encoded features
# Keep a sample for speed if needed
sample_n = min(3000, X_test_enc.shape[0])
X_test_sample = X_test_enc[:sample_n]

explainer_shap = shap.TreeExplainer(rf_model)
shap_values = explainer_shap.shap_values(X_test_sample)

# For binary classification, shap_values may be list-like [class0, class1]
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
else:
    shap_values_pos = shap_values

print('SHAP values shape:', np.array(shap_values_pos).shape)


In [ ]:
# Global summary plot
shap.summary_plot(
    shap_values_pos,
    X_test_sample,
    feature_names=feature_names,
    show=True
)


### SHAP Analysis
After running the summary plot, identify the top 3-4 global features directly from the y-axis (highest mean absolute SHAP values at the top). In many Adult-dataset runs, commonly dominant features include `capital-gain`, `marital-status`, `education-num`, `hours-per-week`, and `age`.

Example relationship interpretation:
- For `hours-per-week`, red points (higher values) often appear more on the positive SHAP side, meaning longer working hours tend to push predictions toward `>50K`.
- For `capital-gain`, very high values usually have strong positive SHAP impact, often among the largest drivers toward the high-income class.

Use your own generated plot as the final source of truth for the exact feature order in your trained model.
